# Colab Session: unlimited-ocr-ec146
Generated from colab-cli history log.

**Session Created**: 2026-07-29 20:09:53
- Endpoint: `gpu-t4-s-kkb-usw1b1-2cquml318qpwe`
- Hardware: `T4`

### Automation: drivemount (2026-07-29 20:09:57)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Session Created**: 2026-07-29 20:12:26
- Endpoint: `gpu-t4-s-kkb-usw1b2-2hwhz5c400728`
- Hardware: `T4`

In [ ]:
"""Install the model's pinned dependencies and prepare remote directories."""

from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import subprocess
import sys

PACKAGES = {
    "torch": "2.10.0",
    "torchvision": "0.25.0",
    "transformers": "4.57.1",
    "Pillow": "12.1.1",
    "matplotlib": "3.10.8",
    "einops": "0.8.2",
    "addict": "2.4.0",
    "easydict": "1.13",
    "PyMuPDF": "1.27.2.2",
    "psutil": "7.2.2",
}


def packages_to_install() -> list[str]:
    missing_or_mismatched: list[str] = []
    for package, expected_version in PACKAGES.items():
        try:
            installed_version = version(package)
        except PackageNotFoundError:
            installed_version = None
        if installed_version != expected_version:
            missing_or_mismatched.append(f"{package}=={expected_version}")
    return missing_or_mismatched


required = packages_to_install()
if required:
    print("Installing: " + ", ".join(required))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", *required],
        check=True,
    )
else:
    print("Pinned dependencies are already installed.")

Path("/content/input").mkdir(parents=True, exist_ok=True)
Path("/content/output").mkdir(parents=True, exist_ok=True)

import torch  # noqa: E402
import transformers  # noqa: E402

print(f"torch={torch.__version__}")
print(f"transformers={transformers.__version__}")
print("Colab environment is ready.")


Installing: torch==2.10.0, torchvision==0.25.0, transformers==4.57.1, Pillow==12.1.1, matplotlib==3.10.8, addict==2.4.0, PyMuPDF==1.27.2.2, psutil==7.2.2


torch=2.10.0+cu128
transformers=4.57.1
Colab environment is ready.


*File Operation*: `upload` on `/content/input/sapl-emenda_146.pdf`

In [ ]:
"""Run baidu/Unlimited-OCR on an image or PDF in Google Colab.

Expected Colab paths:
  input:  /content/input/<file>
  output: /content/output/result.md
"""

from __future__ import annotations

import builtins
import os
import shutil
import tempfile
import time
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import fitz
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "baidu/Unlimited-OCR"
MODEL_REVISION = "07dea832e22aefee32ad281d4b80551282e1c168"
MODEL_PATH_FILE = Path("/content/unlimited_ocr_model_path.txt")
KERNEL_CACHE_KEY = "_unlimited_ocr_runtime_cache_v1"
INPUT_DIR = Path(os.environ.get("OCR_INPUT_DIR", "/content/input"))
OUTPUT_DIR = Path(os.environ.get("OCR_OUTPUT_DIR", "/content/output"))
SUPPORTED_IMAGES = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}


def find_input() -> Path:
    candidates = sorted(
        path
        for path in INPUT_DIR.iterdir()
        if path.is_file() and (path.suffix.lower() == ".pdf" or path.suffix.lower() in SUPPORTED_IMAGES)
    )
    if not candidates:
        raise FileNotFoundError(
            f"No PDF or supported image found in {INPUT_DIR}. "
            f"Supported images: {', '.join(sorted(SUPPORTED_IMAGES))}"
        )
    if len(candidates) > 1:
        raise RuntimeError(
            f"Expected one input file in {INPUT_DIR}, found: "
            + ", ".join(path.name for path in candidates)
        )
    return candidates[0]


def pdf_to_images(pdf_path: Path, dpi: int = 200) -> tuple[list[str], Path]:
    temp_dir = Path(tempfile.mkdtemp(prefix="unlimited_ocr_pdf_"))
    document = fitz.open(pdf_path)
    matrix = fitz.Matrix(dpi / 72, dpi / 72)
    paths: list[str] = []
    try:
        for page_number, page in enumerate(document, start=1):
            output_path = temp_dir / f"page_{page_number:04d}.png"
            page.get_pixmap(matrix=matrix, alpha=False).save(output_path)
            paths.append(str(output_path))
    finally:
        document.close()
    return paths, temp_dir


def collect_markdown(model_output_dir: Path) -> str:
    parts: list[str] = []
    for path in sorted(model_output_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8").strip()
            if text:
                parts.append(text)
    if not parts:
        raise RuntimeError(f"The model produced no Markdown or text files in {model_output_dir}")
    return "\n\n".join(parts)


def infer_single(
    model: object,
    tokenizer: object,
    image_path: str,
    output_path: Path,
    *,
    use_crop_mode: bool,
) -> None:
    output_path.mkdir(parents=True, exist_ok=True)
    model.infer(
        tokenizer,
        prompt="<image>document parsing.",
        image_file=image_path,
        output_path=str(output_path),
        base_size=1024,
        image_size=640 if use_crop_mode else 1024,
        crop_mode=use_crop_mode,
        max_length=32768,
        no_repeat_ngram_size=35,
        ngram_window=128,
        save_results=True,
    )


def resolve_model_source() -> tuple[str, bool]:
    configured_path = os.environ.get("UNLIMITED_OCR_MODEL_PATH")
    if configured_path:
        model_path = Path(configured_path)
    elif MODEL_PATH_FILE.is_file():
        model_path = Path(MODEL_PATH_FILE.read_text(encoding="utf-8").strip())
    else:
        return MODEL_NAME, False

    if not (model_path / "config.json").is_file():
        raise RuntimeError(f"Configured local model snapshot is invalid: {model_path}")
    return str(model_path), True


def load_model() -> tuple[object, object, bool]:
    model_source, is_local = resolve_model_source()
    cached = getattr(builtins, KERNEL_CACHE_KEY, None)
    if (
        isinstance(cached, dict)
        and cached.get("revision") == MODEL_REVISION
        and cached.get("source") == model_source
    ):
        model = cached["model"]
        tokenizer = cached["tokenizer"]
        if next(model.parameters()).is_cuda:
            return model, tokenizer, True

    print(f"Loading {MODEL_NAME} from {model_source}...")
    load_started = time.perf_counter()
    common_options = {
        "trust_remote_code": True,
        "local_files_only": is_local,
    }
    if not is_local:
        common_options["revision"] = MODEL_REVISION
    tokenizer = AutoTokenizer.from_pretrained(model_source, **common_options)
    model = AutoModel.from_pretrained(
        model_source,
        use_safetensors=True,
        torch_dtype=torch.bfloat16,
        **common_options,
    ).eval().cuda()
    setattr(
        builtins,
        KERNEL_CACHE_KEY,
        {
            "model": model,
            "tokenizer": tokenizer,
            "revision": MODEL_REVISION,
            "source": model_source,
        },
    )
    print(f"Model loaded onto GPU in {time.perf_counter() - load_started:.1f}s")
    return model, tokenizer, False


def main() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required.")

    input_path = find_input()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model_output_dir = OUTPUT_DIR / "model_files"
    if model_output_dir.exists():
        shutil.rmtree(model_output_dir)
    model_output_dir.mkdir(parents=True, exist_ok=True)

    gpu_name = torch.cuda.get_device_name(0)
    compute_capability = torch.cuda.get_device_capability(0)
    print(f"GPU: {gpu_name} (compute capability {compute_capability[0]}.{compute_capability[1]})")
    print("Model dtype: torch.bfloat16")
    print(f"Input: {input_path}")
    model, tokenizer, reused_model = load_model()
    print("Model cache: " + ("GPU memory hit" if reused_model else "loaded into GPU"))

    temp_dir: Path | None = None
    inference_started = time.perf_counter()
    try:
        if input_path.suffix.lower() == ".pdf":
            image_paths, temp_dir = pdf_to_images(input_path)
            print(f"PDF pages: {len(image_paths)}")
            if compute_capability[0] >= 8:
                model.infer_multi(
                    tokenizer,
                    prompt="<image>Multi page parsing.",
                    image_files=image_paths,
                    output_path=str(model_output_dir),
                    image_size=1024,
                    max_length=32768,
                    no_repeat_ngram_size=35,
                    ngram_window=1024,
                    save_results=True,
                )
            else:
                print("T4 memory mode: processing the PDF one page at a time.")
                for page_number, image_path in enumerate(image_paths, start=1):
                    print(f"Page {page_number}/{len(image_paths)}")
                    infer_single(
                        model,
                        tokenizer,
                        image_path,
                        model_output_dir / f"page_{page_number:04d}",
                        use_crop_mode=False,
                    )
        else:
            infer_single(
                model,
                tokenizer,
                str(input_path),
                model_output_dir,
                use_crop_mode=compute_capability[0] >= 8,
            )

        result_path = OUTPUT_DIR / "result.md"
        result_path.write_text(collect_markdown(model_output_dir) + "\n", encoding="utf-8")
        print(f"Result: {result_path}")
        print(f"Inference time: {time.perf_counter() - inference_started:.1f}s")
    finally:
        if temp_dir is not None:
            shutil.rmtree(temp_dir, ignore_errors=True)


if __name__ == "__main__":
    main()


GPU: Tesla T4 (compute capability 7.5)
Model dtype: torch.bfloat16
Input: /content/input/sapl-emenda_146.pdf
Loading baidu/Unlimited-OCR from baidu/Unlimited-OCR...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

...

que 

se 

refere 

o 

KeyboardInterrupt: 

*File Operation*: `ls` on `/content/output/model_files`

In [ ]:
"""Run baidu/Unlimited-OCR on an image or PDF in Google Colab.

Expected Colab paths:
  input:  /content/input/<file>
  output: /content/output/result.md
"""

from __future__ import annotations

import builtins
import os
import shutil
import tempfile
import time
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import fitz
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "baidu/Unlimited-OCR"
MODEL_REVISION = "07dea832e22aefee32ad281d4b80551282e1c168"
MODEL_PATH_FILE = Path("/content/unlimited_ocr_model_path.txt")
KERNEL_CACHE_KEY = "_unlimited_ocr_runtime_cache_v1"
INPUT_DIR = Path(os.environ.get("OCR_INPUT_DIR", "/content/input"))
OUTPUT_DIR = Path(os.environ.get("OCR_OUTPUT_DIR", "/content/output"))
MAX_LENGTH = int(os.environ.get("OCR_MAX_LENGTH", "4096"))
SUPPORTED_IMAGES = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}


def find_input() -> Path:
    candidates = sorted(
        path
        for path in INPUT_DIR.iterdir()
        if path.is_file() and (path.suffix.lower() == ".pdf" or path.suffix.lower() in SUPPORTED_IMAGES)
    )
    if not candidates:
        raise FileNotFoundError(
            f"No PDF or supported image found in {INPUT_DIR}. "
            f"Supported images: {', '.join(sorted(SUPPORTED_IMAGES))}"
        )
    if len(candidates) > 1:
        raise RuntimeError(
            f"Expected one input file in {INPUT_DIR}, found: "
            + ", ".join(path.name for path in candidates)
        )
    return candidates[0]


def pdf_to_images(pdf_path: Path, dpi: int = 200) -> tuple[list[str], Path]:
    temp_dir = Path(tempfile.mkdtemp(prefix="unlimited_ocr_pdf_"))
    document = fitz.open(pdf_path)
    matrix = fitz.Matrix(dpi / 72, dpi / 72)
    paths: list[str] = []
    try:
        for page_number, page in enumerate(document, start=1):
            output_path = temp_dir / f"page_{page_number:04d}.png"
            page.get_pixmap(matrix=matrix, alpha=False).save(output_path)
            paths.append(str(output_path))
    finally:
        document.close()
    return paths, temp_dir


def collect_markdown(model_output_dir: Path) -> str:
    parts: list[str] = []
    for path in sorted(model_output_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in {".md", ".txt"}:
            text = path.read_text(encoding="utf-8").strip()
            if text:
                parts.append(text)
    if not parts:
        raise RuntimeError(f"The model produced no Markdown or text files in {model_output_dir}")
    return "\n\n".join(parts)


def infer_single(
    model: object,
    tokenizer: object,
    image_path: str,
    output_path: Path,
    *,
    use_crop_mode: bool,
) -> None:
    output_path.mkdir(parents=True, exist_ok=True)
    model.infer(
        tokenizer,
        prompt="<image>document parsing.",
        image_file=image_path,
        output_path=str(output_path),
        base_size=1024,
        image_size=640 if use_crop_mode else 1024,
        crop_mode=use_crop_mode,
        max_length=MAX_LENGTH,
        no_repeat_ngram_size=35,
        ngram_window=128,
        save_results=True,
    )


def resolve_model_source() -> tuple[str, bool]:
    configured_path = os.environ.get("UNLIMITED_OCR_MODEL_PATH")
    if configured_path:
        model_path = Path(configured_path)
    elif MODEL_PATH_FILE.is_file():
        model_path = Path(MODEL_PATH_FILE.read_text(encoding="utf-8").strip())
    else:
        return MODEL_NAME, False

    if not (model_path / "config.json").is_file():
        raise RuntimeError(f"Configured local model snapshot is invalid: {model_path}")
    return str(model_path), True


def load_model() -> tuple[object, object, bool]:
    model_source, is_local = resolve_model_source()
    cached = getattr(builtins, KERNEL_CACHE_KEY, None)
    if (
        isinstance(cached, dict)
        and cached.get("revision") == MODEL_REVISION
        and cached.get("source") == model_source
    ):
        model = cached["model"]
        tokenizer = cached["tokenizer"]
        if next(model.parameters()).is_cuda:
            return model, tokenizer, True

    print(f"Loading {MODEL_NAME} from {model_source}...")
    load_started = time.perf_counter()
    common_options = {
        "trust_remote_code": True,
        "local_files_only": is_local,
    }
    if not is_local:
        common_options["revision"] = MODEL_REVISION
    tokenizer = AutoTokenizer.from_pretrained(model_source, **common_options)
    model = AutoModel.from_pretrained(
        model_source,
        use_safetensors=True,
        torch_dtype=torch.bfloat16,
        **common_options,
    ).eval().cuda()
    setattr(
        builtins,
        KERNEL_CACHE_KEY,
        {
            "model": model,
            "tokenizer": tokenizer,
            "revision": MODEL_REVISION,
            "source": model_source,
        },
    )
    print(f"Model loaded onto GPU in {time.perf_counter() - load_started:.1f}s")
    return model, tokenizer, False


def main() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required.")

    input_path = find_input()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model_output_dir = OUTPUT_DIR / "model_files"
    if model_output_dir.exists():
        shutil.rmtree(model_output_dir)
    model_output_dir.mkdir(parents=True, exist_ok=True)

    gpu_name = torch.cuda.get_device_name(0)
    compute_capability = torch.cuda.get_device_capability(0)
    print(f"GPU: {gpu_name} (compute capability {compute_capability[0]}.{compute_capability[1]})")
    print("Model dtype: torch.bfloat16")
    print(f"Input: {input_path}")
    model, tokenizer, reused_model = load_model()
    print("Model cache: " + ("GPU memory hit" if reused_model else "loaded into GPU"))

    temp_dir: Path | None = None
    inference_started = time.perf_counter()
    try:
        if input_path.suffix.lower() == ".pdf":
            image_paths, temp_dir = pdf_to_images(input_path)
            print(f"PDF pages: {len(image_paths)}")
            if compute_capability[0] >= 8:
                model.infer_multi(
                    tokenizer,
                    prompt="<image>Multi page parsing.",
                    image_files=image_paths,
                    output_path=str(model_output_dir),
                    image_size=1024,
                    max_length=MAX_LENGTH,
                    no_repeat_ngram_size=35,
                    ngram_window=1024,
                    save_results=True,
                )
            else:
                print("T4 memory mode: processing the PDF one page at a time.")
                for page_number, image_path in enumerate(image_paths, start=1):
                    print(f"Page {page_number}/{len(image_paths)}")
                    infer_single(
                        model,
                        tokenizer,
                        image_path,
                        model_output_dir / f"page_{page_number:04d}",
                        use_crop_mode=False,
                    )
        else:
            infer_single(
                model,
                tokenizer,
                str(input_path),
                model_output_dir,
                use_crop_mode=compute_capability[0] >= 8,
            )

        result_path = OUTPUT_DIR / "result.md"
        result_path.write_text(collect_markdown(model_output_dir) + "\n", encoding="utf-8")
        print(f"Result: {result_path}")
        print(f"Inference time: {time.perf_counter() - inference_started:.1f}s")
    finally:
        if temp_dir is not None:
            shutil.rmtree(temp_dir, ignore_errors=True)


if __name__ == "__main__":
    main()


GPU: Tesla T4 (compute capability 7.5)
Model dtype: torch.bfloat16
Input: /content/input/sapl-emenda_146.pdf
Loading baidu/Unlimited-OCR from baidu/Unlimited-OCR...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


`torch_dtype` is deprecated! Use `dtype` instead!


Some weights of UnlimitedOCRForCausalLM were not initialized from the model checkpoint at baidu/Unlimited-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded onto GPU in 41.4s
Model cache: loaded into GPU


...

image:   0%|          | 0/1 [00:00<?, ?it/s]

image: 100%|██████████| 1/1 [00:00<00:00, 10407.70it/s]

other:   0%|          | 0/7 [00:00<?, ?it/s]

other: 100%|██████████| 7/7 [00:00<00:00, 80000.35it/s]

Result: /content/output/result.md
Inference time: 494.1s


*File Operation*: `download` on `/content/output/result.md`